# Etapa 2 CRISP-DM — Comprensión de los datos
## Proyecto 09: reingreso hospitalario en pacientes con diabetes

**UTEC · Proyecto Integrador I · Grupo 2**

Este notebook cubre las tareas 2.1 a 2.4: recopilar, describir, explorar y
verificar la calidad. Es un análisis descriptivo y exploratorio. `df_parquet` conserva todos
los episodios originales y `df_cohort` es solamente la vista exploratoria que
separa los destinos de alta que indican fallecimiento.

> **Cuaderno autónomo.** No requiere el repositorio ni archivos `.py`: solo librerías y la celda de **parámetros**. Al final (§2.5) publica su resultado como **contrato** en MLflow, MinIO y GitHub, y E3 consume ese contrato.

## Etapas

| Tarea | Salida en este notebook |
|---|---|
| 2.1 Recopilar | fuentes, acceso y problemas observados |
| 2.2 Describir | unidad de análisis, volumen, variables y resultado |
| 2.3 Explorar | distribuciones, relaciones, siete tipos de gráficos y tres hallazgos |
| 2.4 Verificar | faltantes, códigos, duplicados, cardinalidad, atípicos y representatividad |

El material de referencia es `Clases/crispdm_e1_e2.md`. La procedencia de los
datos es Diabetes 130-US Hospitals for years 1999–2008, UCI, dataset 296.
| 2.5 Publicar *(exigencia del curso)* | contrato E2→E3 en MLflow (`tipo=eda`), MinIO (`datasets/crudo`, `datasets/eda`) y GitHub (`reportes/e2/`) |

# Configuración inicial

In [ ]:
# =============================== PARÁMETROS DE E2 ===============================
# Único lugar que se edita. Este cuaderno es AUTÓNOMO: no necesita el repositorio
# ni archivos .py; solo librerías (pandas, boto3, mlflow). Corre igual en Colab.
#
# Si el repositorio de GitHub tiene proyecto.yaml, sus valores (MinIO, MLflow,
# fuentes) reemplazan automáticamente a los de CONFIG: la configuración vive en GitHub.

GITHUB_URL  = "https://github.com/CAMBIAR-USUARIO/p09-reingreso-diabetes.git"
GITHUB_RAMA = "main"
GITHUB_MODO = "pr"          # "pr": rama e2/<tag> + pull request (rama protegida) · "directo": commit en GITHUB_RAMA
TAG_CRUDO   = "crudo-v1"    # versión del crudo que se publica (crudo-v2 si el crudo cambia)
GITHUB_SUBCARPETA = ""      # carpeta del proyecto dentro del repo ("" = raíz). Ej.: "p09-reingreso-diabetes"

PUBLICAR_MLFLOW = True      # run tipo=eda (raíz del linaje)
PUBLICAR_MINIO  = True      # datasets/crudo/<tag>/ y datasets/eda/<tag>/ + catálogo
PUBLICAR_GITHUB = True      # reportes/e2/ (contrato que consume E3)
FORZAR_NUEVO_RUN = False    # True: crea otro run aunque el crudo ya esté publicado

CONFIG = {
    "minio":  {"endpoint": "http://208.68.37.250:9000", "bucket": "p09-reingreso", "prefijo_datasets": "datasets"},
    "mlflow": {"tracking_uri": "http://208.68.37.250:5000", "experimento": "p09-reingreso-diabetes"},
    "fuentes": {
        "parquet": "https://drive.google.com/uc?export=download&id=122SIHS0YFXUi6K1x2D2cVP_iOLtk9oZT",
        "ids_mapping": "https://drive.google.com/uc?export=download&id=136U7krs2lICcTFURWN2bxHJSwJ8NP771",
    },
    "seleccion": {"disposiciones_fallecimiento": [11, 19, 20, 21]},
}
# SECRETOS: nunca en este cuaderno. Orden de búsqueda:
#   1) variable de entorno  2) Secretos de Colab (icono de llave)  3) se piden por teclado
# En Colab: crear MINIO_ACCESS_KEY, MINIO_SECRET_KEY y GITHUB_TOKEN y ACTIVAR
# «Acceso al notebook» en cada uno (sin ese interruptor el notebook no los ve).
# GITHUB_TOKEN: token fine-grained del repo con Contents y Pull requests en read/write.

### Credenciales

Dos formas, en este orden de preferencia: **Secretos de Colab** (icono de llave, con «Acceso al notebook» activado) o, si se prefiere la vía directa, escribirlas en la celda siguiente — que **no debe subirse con valores reales**.

In [ ]:
# ============================== CREDENCIALES ==================================
# ⚠  ESTA CELDA CONTIENE SECRETOS. Reglas:
#    · Rellenar los valores SOLO en la copia que se ejecuta (Colab / máquina local).
#    · NUNCA guardar el cuaderno con valores reales en GitHub ni en Drive compartido.
#      Antes de subirlo: vaciar esta celda (dejar los "" ) o usar Secretos de Colab.
#    · Si un valor llegó a un repositorio o a un chat, borrarlo NO basta: hay que
#      REVOCARLO y generar otro (token de GitHub) o rotar la clave (MinIO).
#    · La celda de publicación avisa si el cuaderno trae secretos escritos.
#
# Alternativa recomendada (sin secretos en el archivo): dejar esto vacío y crear
# MINIO_ACCESS_KEY, MINIO_SECRET_KEY y GITHUB_TOKEN en los Secretos de Colab
# (icono de llave), activando «Acceso al notebook» en cada uno.

MINIO_ACCESS_KEY = ''     # p. ej. "p09-equipo"
MINIO_SECRET_KEY = ''     # p. ej. "********"
GITHUB_TOKEN     = ''     # p. ej. "github_pat_..."  (permisos Contents y Pull requests: read/write)

In [ ]:
import io
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Markdown, display

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_rows", 100)

In [ ]:
import base64
import hashlib
import json
import os
import subprocess
import sys
import urllib.error
import urllib.request
from pathlib import Path

# --- Sobrescrituras por variable de entorno (las usa `python tareas.py e2` al
#     ejecutar este cuaderno sin interfaz desde el repositorio)
GITHUB_URL = os.environ.get("GITHUB_URL", GITHUB_URL)
for var, flag in [("P09_PUBLICAR_GITHUB", "PUBLICAR_GITHUB"), ("P09_PUBLICAR_MINIO", "PUBLICAR_MINIO"),
                  ("P09_PUBLICAR_MLFLOW", "PUBLICAR_MLFLOW")]:
    if var in os.environ:
        globals()[flag] = os.environ[var] == "1"
TAG_CRUDO = os.environ.get("P09_TAG_CRUDO", TAG_CRUDO)


DIAG_SECRETOS = {}


def secreto(nombre, pedir=False):
    """Celda CREDENCIALES -> variable de entorno -> Secretos de Colab -> teclado.
    Nunca se imprime el valor, solo de dónde salió."""
    directo = globals().get(nombre)
    if isinstance(directo, str) and directo.strip() and not directo.startswith(("p. ej.", "github_pat_...")):
        DIAG_SECRETOS[nombre] = "celda CREDENCIALES del cuaderno (no subir el archivo con este valor)"
        return directo.strip()
    if os.environ.get(nombre):
        DIAG_SECRETOS.setdefault(nombre, "variable de entorno")
        return os.environ[nombre]
    try:
        from google.colab import userdata
        try:
            valor = userdata.get(nombre)
            if valor:
                os.environ[nombre] = valor            # se cachea: no se vuelve a consultar
                DIAG_SECRETOS[nombre] = "Secretos de Colab"
                return valor
            DIAG_SECRETOS[nombre] = "existe en Secretos de Colab pero está vacío"
        except Exception as e:
            tipo = type(e).__name__
            DIAG_SECRETOS[nombre] = {
                "SecretNotFoundError": "no existe en Secretos de Colab (crearlo en el icono de llave)",
                "NotebookAccessError": "existe, pero falta activar «Acceso al notebook»",
            }.get(tipo, f"Colab no lo entregó ({tipo})")
    except ImportError:
        DIAG_SECRETOS[nombre] = "no definido (fuera de Colab: use variable de entorno)"
    if pedir:
        try:
            from getpass import getpass
            valor = getpass(f"{nombre} ({DIAG_SECRETOS[nombre]}). Escríbalo o deje vacío para omitir: ").strip()
            if valor:
                os.environ[nombre] = valor
                DIAG_SECRETOS[nombre] = "ingresado por teclado (solo esta sesión)"
                return valor
        except Exception:                             # ejecución sin interfaz: no hay teclado
            pass
    return None


def asegurar(paquete, modulo=None):
    """Instala una LIBRERÍA si falta (no hay código del proyecto que instalar)."""
    try:
        return __import__(modulo or paquete)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", paquete])
        return __import__(modulo or paquete)


def slug_github(url):
    m = re.search(r"github\.com[:/]([^/]+)/([^/]+?)(?:\.git)?/?$", url or "")
    return f"{m.group(1)}/{m.group(2)}" if m else None


def leer_config_github():
    """Trae proyecto.yaml del repositorio (raw) para no duplicar configuración."""
    slug = slug_github(GITHUB_URL)
    if not slug or "CAMBIAR-USUARIO" in slug:
        return None
    sub = f"{GITHUB_SUBCARPETA.strip('/')}/" if GITHUB_SUBCARPETA.strip("/") else ""
    url_yaml = f"https://raw.githubusercontent.com/{slug}/{GITHUB_RAMA}/{sub}proyecto.yaml"
    req = urllib.request.Request(url_yaml)
    if secreto("GITHUB_TOKEN"):
        req.add_header("Authorization", f"token {secreto('GITHUB_TOKEN')}")
    try:
        import yaml
        with urllib.request.urlopen(req, timeout=20) as r:
            return yaml.safe_load(r.read())
    except urllib.error.HTTPError as e:
        motivo = "no existe en esa ruta, o el repo es privado y falta GITHUB_TOKEN" if e.code == 404 else f"HTTP {e.code}"
        print(f"(proyecto.yaml no leído de {url_yaml}: {motivo}. Se usan los valores de CONFIG)")
    except Exception as e:
        print(f"(proyecto.yaml no leído: {type(e).__name__}. Se usan los valores de CONFIG)")
    return None


cfg_github = leer_config_github()
if cfg_github:
    for seccion in ("minio", "mlflow", "fuentes"):
        CONFIG[seccion].update(cfg_github.get(seccion) or {})
    print("Configuración leída de GitHub: proyecto.yaml")
CONFIG["minio"]["endpoint"] = os.environ.get("MINIO_ENDPOINT", CONFIG["minio"]["endpoint"])
CONFIG["mlflow"]["tracking_uri"] = os.environ.get("MLFLOW_TRACKING_URI", CONFIG["mlflow"]["tracking_uri"])

# Carpeta del contrato: en Colab, local al entorno; desde el repo, reportes/e2
SALIDA = Path(os.environ.get("P09_SALIDA", "p09_e2_salida/reportes/e2")).resolve()
(SALIDA / "figuras").mkdir(parents=True, exist_ok=True)

print(f"GitHub : {GITHUB_URL} ({GITHUB_RAMA}, modo {GITHUB_MODO})")
print(f"MinIO  : {CONFIG['minio']['endpoint']} · bucket {CONFIG['minio']['bucket']}")
print(f"MLflow : {CONFIG['mlflow']['tracking_uri']} · {CONFIG['mlflow']['experimento']}")
print(f"Salida : {SALIDA}")

### Verificación previa de los destinos

Si algo falla, se detiene aquí con el motivo exacto, antes del análisis.

In [ ]:
# --- Verificación previa: falla AQUÍ, en segundos, y no al final del análisis --------
import pandas as pd
pd.set_option("display.max_colwidth", 200)   # que no se corte la columna 'detalle'
CONTINUAR_AUNQUE_FALLE = False      # True: seguir con el análisis aunque algún destino falle

if any(DIAG_SECRETOS.get(n, "").startswith("celda CREDENCIALES") for n in ("MINIO_ACCESS_KEY", "GITHUB_TOKEN")):
    pass   # el aviso se emite al final, junto al resumen de publicación

estado = []
if PUBLICAR_MINIO:
    acceso, clave = secreto("MINIO_ACCESS_KEY", pedir=True), secreto("MINIO_SECRET_KEY", pedir=True)
    if not (acceso and clave):
        estado.append(["MinIO", "❌", f"faltan credenciales: MINIO_ACCESS_KEY {DIAG_SECRETOS.get('MINIO_ACCESS_KEY')}; "
                                     f"MINIO_SECRET_KEY {DIAG_SECRETOS.get('MINIO_SECRET_KEY')}"])
    else:
        boto3 = asegurar("boto3")
        from botocore.config import Config as BotoConfig
        s3 = boto3.client("s3", endpoint_url=CONFIG["minio"]["endpoint"], region_name="us-east-1",
                          aws_access_key_id=acceso, aws_secret_access_key=clave,
                          config=BotoConfig(signature_version="s3v4", s3={"addressing_style": "path"},
                                            connect_timeout=8, read_timeout=60, retries={"max_attempts": 1}))
        try:
            buckets = {b["Name"] for b in s3.list_buckets().get("Buckets", [])}
            existe = CONFIG["minio"]["bucket"] in buckets
            estado.append(["MinIO", "✅", f"{CONFIG['minio']['endpoint']} responde · {len(buckets)} buckets · "
                                         f"'{CONFIG['minio']['bucket']}' {'existe' if existe else 'se creará'}"])
        except Exception as e:
            txt = f"{type(e).__name__}: {str(e)[:160]}"
            pista = ("credenciales rechazadas" if any(k in txt for k in ("InvalidAccessKeyId", "SignatureDoesNotMatch", "AccessDenied"))
                     else "no responde: ¿IP/puerto? La API S3 suele ser :9000 (el :9001 es la consola web)"
                     if any(k in txt for k in ("EndpointConnectionError", "ConnectTimeout", "Connection"))
                     else "respuesta inesperada: ¿apunta a la consola (:9001) en vez de la API (:9000)?")
            estado.append(["MinIO", "❌", f"{pista} · {txt}"])
if PUBLICAR_MLFLOW:
    try:
        with urllib.request.urlopen(f"{CONFIG['mlflow']['tracking_uri'].rstrip('/')}/health", timeout=10) as r:
            estado.append(["MLflow", "✅", f"{CONFIG['mlflow']['tracking_uri']} responde ({r.status})"])
    except Exception as e:
        estado.append(["MLflow", "❌" if CONFIG["mlflow"]["tracking_uri"].startswith("http") else "✅",
                       f"{CONFIG['mlflow']['tracking_uri']} · {type(e).__name__}"])
if PUBLICAR_GITHUB:
    token, slug = secreto("GITHUB_TOKEN", pedir=True), slug_github(GITHUB_URL)
    if not slug or "CAMBIAR-USUARIO" in GITHUB_URL:
        estado.append(["GitHub", "❌", "GITHUB_URL sin configurar en PARÁMETROS"])
    elif not token:
        estado.append(["GitHub", "❌", f"falta GITHUB_TOKEN: {DIAG_SECRETOS.get('GITHUB_TOKEN')}"])
    else:
        api = os.environ.get("P09_GITHUB_API", "https://api.github.com")
        cab = {"Authorization": f"Bearer {token}", "Accept": "application/vnd.github+json",
               "X-GitHub-Api-Version": "2022-11-28"}

        def api_get(ruta):
            """Devuelve (codigo, cuerpo). No lanza: el diagnóstico necesita ver el error."""
            try:
                with urllib.request.urlopen(urllib.request.Request(f"{api}{ruta}", headers=cab), timeout=15) as r:
                    return r.status, json.loads(r.read() or b"{}")
            except urllib.error.HTTPError as e:
                cuerpo = {}
                try:
                    cuerpo = json.loads(e.read() or b"{}")
                except Exception:
                    pass
                return e.code, cuerpo
            except Exception as e:
                return None, {"message": f"{type(e).__name__}: {e}"}

        cod_user, quien = api_get("/user")
        if cod_user == 200:
            print(f"Token válido · usuario: {quien.get('login')}")
        elif cod_user == 403 and "fine-grained" in str(quien.get("message", "")):
            pass                       # algunos tokens fine-grained no pueden leer /user: es normal
        elif cod_user == 401:
            estado.append(["GitHub", "❌", "GITHUB_TOKEN inválido, mal copiado o REVOCADO. "
                                           "Genere otro en github.com/settings/personal-access-tokens"])

        if not any(e[0] == "GitHub" for e in estado):
            cod, repo = api_get(f"/repos/{slug}")
            if cod == 200:
                puede = (repo.get("permissions") or {}).get("push")
                defecto, vacio = repo.get("default_branch"), repo.get("size", 1) == 0
                notas = [slug, f"rama por defecto: {defecto}"]
                if vacio:
                    notas.append("repositorio SIN commits: se creará el commit inicial (sin PR)")
                elif defecto and defecto != GITHUB_RAMA:
                    notas.append(f"GITHUB_RAMA = '{GITHUB_RAMA}' no existe o no coincide; se usará '{defecto}'")
                if puede is False:
                    notas.append("el token NO tiene permiso de escritura (Contents: Read and write)")
                estado.append(["GitHub", "✅" if puede is not False else "❌", " · ".join(notas)])
            else:
                motivo = {
                    401: "token inválido, mal copiado o revocado",
                    403: ("acceso prohibido: si la organización usa SSO, hay que autorizar el token; "
                          "si es fine-grained, puede estar pendiente de aprobación de un administrador"),
                    404: (f"el token no ve '{slug}'. Causas: el nombre del repo no es ese; el token "
                          "fine-grained no incluyó ese repositorio en 'Only select repositories'; "
                          "la organización aún no aprobó el token; o el token es de otra cuenta"),
                }.get(cod, f"HTTP {cod}")
                detalle_api = quien.get("message") if cod is None else repo.get("message", "")
                estado.append(["GitHub", "❌", f"{motivo}" + (f" · GitHub dice: {detalle_api}" if detalle_api else "")])

tabla_estado = pd.DataFrame(estado, columns=["destino", "estado", "detalle"])
display(tabla_estado) if len(tabla_estado) else print("No hay destinos de publicación activados.")
fallos = tabla_estado[tabla_estado["estado"] == "❌"]
if len(fallos):
    print("\nMOTIVO DEL FALLO:")
    for f in fallos.itertuples():
        print(f"  · {f.destino}: {f.detalle}")
if len(fallos) and not CONTINUAR_AUNQUE_FALLE:
    raise RuntimeError(" | ".join(f"{f.destino}: {f.detalle}" for f in fallos.itertuples())
                       + "  ——  Corrija lo anterior (o desactive ese PUBLICAR_* / ponga "
                         "CONTINUAR_AUNQUE_FALLE = True) y vuelva a ejecutar esta celda.")

# 2.1 · Recopilar los datos iniciales

Las fuentes de trabajo son los dos archivos compartidos en Google Drive. Se
mantienen aquí las URL entregadas por el equipo y los nombres `df_parquet` y
`df_mapping`. El CSV de códigos es concatenado: contiene tres bloques con
cabeceras distintas, por lo que primero se lee como dos columnas y después se
separa por cabecera. No se debe tratar todo el CSV como un único diccionario.

In [ ]:
# Fuentes compartidas por el equipo (Drive). Se leen como BYTES para calcular su
# huella md5: es la identidad del crudo que viaja a MinIO, MLflow, GitHub y E3.
ruta_parquet = CONFIG["fuentes"]["parquet"]
ruta_ids_mapping = CONFIG["fuentes"]["ids_mapping"]


def leer_bytes(origen, variable_local):
    local = os.environ.get(variable_local)          # lo usa `tareas.py e2` (copia local)
    if local:
        return Path(local).read_bytes()
    url = origen + ("&confirm=t" if "drive.google.com" in origen else "")
    with urllib.request.urlopen(url, timeout=180) as r:
        datos = r.read()
    if datos[:200].lstrip().lower().startswith((b"<!doctype", b"<html")):
        raise RuntimeError(f"Drive devolvió una página HTML: revise que {origen} sea público")
    return datos


bytes_parquet = leer_bytes(ruta_parquet, "P09_CRUDO_LOCAL")
bytes_mapping = leer_bytes(ruta_ids_mapping, "P09_MAPPING_LOCAL")
md5_crudo = hashlib.md5(bytes_parquet).hexdigest()
md5_mapping = hashlib.md5(bytes_mapping).hexdigest()

df_parquet = pd.read_parquet(io.BytesIO(bytes_parquet))
df_mapping = pd.read_csv(
    io.BytesIO(bytes_mapping),
    header=None,
    names=["codigo", "description"],
    dtype="string",
    keep_default_na=False,
)

print(f"Parquet: {df_parquet.shape[0]:,} filas y {df_parquet.shape[1]} columnas · md5 {md5_crudo}")
print(f"CSV de códigos: {len(df_mapping):,} líneas, incluidos separadores · md5 {md5_mapping}")
display(df_parquet.head(3))

In [ ]:
# El CSV de Drive trae tres bloques: una cabecera identifica cada variable.
variables_codificadas = {
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id",
}

def separar_mapping(tabla):
    bloques = []
    variable_actual = None
    for _, fila in tabla.iterrows():
        codigo = str(fila["codigo"]).strip()
        descripcion = str(fila["description"]).strip()
        if codigo in variables_codificadas and descripcion == "description":
            variable_actual = codigo
        elif variable_actual and codigo:
            bloques.append(
                {
                    "variable": variable_actual,
                    "codigo": int(codigo),
                    "description": descripcion,
                }
            )
    return pd.DataFrame(bloques)

mapping_table = separar_mapping(df_mapping)
assert set(mapping_table["variable"]) == variables_codificadas
print("Bloques interpretados:", mapping_table.groupby("variable").size().to_dict())
display(mapping_table.head(10))

### Acceso y problemas observados

- El Parquet conserva la tabla de episodios y el CSV conserva los significados
de los códigos administrativos. Ambos se leen directamente desde Drive; si el
acceso no está disponible, la ejecución debe detenerse y reportarse, no
presentar datos locales como si fueran de Drive.
- Una lectura única del CSV desplaza los bloques bajo la primera cabecera. El
parseo anterior conserva los tres nombres (`admission_type_id`,
`discharge_disposition_id` y `admission_source_id`) y convierte sus códigos a
enteros.
- Los archivos no incluyen fecha por episodio ni hospital. Por ello no se
pueden analizar tendencias temporales, tasas diarias o diferencias entre
centros.

# 2.2 · Describir los datos

Una fila representa un episodio de hospitalización (`encounter_id`) de un
paciente identificado por `patient_nbr`; un paciente puede aparecer en varios
episodios. El resultado observado es `readmitted`, con `NO`, `>30` y `<30`.
La documentación de la fuente cubre 1999–2008, pero el archivo no trae una
fecha por fila. El esquema siguiente describe las **50 columnas originales**;
no se agregan variables derivadas al conjunto completo.

In [ ]:
identificador_cols = ["encounter_id", "patient_nbr"]
numericas_cols = [
    "time_in_hospital", "num_lab_procedures", "num_procedures",
    "num_medications", "number_outpatient", "number_emergency",
    "number_inpatient", "number_diagnoses",
]
target_col = "readmitted"
categoricas_cols = [
    c for c in df_parquet.columns
    if c not in identificador_cols + numericas_cols
]
assert len(df_parquet.columns) == 50
assert len(identificador_cols + numericas_cols + categoricas_cols) == 50

schema = pd.DataFrame({
    "variable": df_parquet.columns,
    "tipo pandas": [str(df_parquet[c].dtype) for c in df_parquet.columns],
    "tipo para E2": [
        "identificador técnico" if c in identificador_cols else
        "numérica discreta" if c in numericas_cols else
        "categórica / resultado" if c == target_col else
        "categórica"
        for c in df_parquet.columns
    ],
    "valores distintos": [df_parquet[c].nunique(dropna=True) for c in df_parquet.columns],
    "% faltantes (raw)": [df_parquet[c].isna().mean() * 100 for c in df_parquet.columns],
})
print(f"Unidad: episodio hospitalario | {len(df_parquet):,} filas | {len(schema)} variables")
display(schema.round({"% faltantes (raw)": 2}))

In [ ]:
# Resúmenes compactos para magnitudes y resultado.

resumen_parquet = df_parquet[numericas_cols].describe().T[
    ["count", "mean", "50%", "std", "min", "max"]
].rename(columns={"50%": "median"})
print("Magnitudes numéricas del raw:")
display(resumen_parquet.round(2))

resumen_vategorico = pd.DataFrame({
    "variable": categoricas_cols,
    "moda": [df_parquet[c].mode(dropna=True).iloc[0] if df_parquet[c].notna().any() else pd.NA for c in categoricas_cols],
    "frecuencia moda": [df_parquet[c].value_counts(dropna=True).iloc[0] if df_parquet[c].notna().any() else 0 for c in categoricas_cols],
}).set_index("variable")
print("Primeras variables categóricas (el esquema anterior contiene las 50):")
display(resumen_vategorico.head(12))
print("Distribución de readmitted en el raw:")
display(df_parquet[target_col].value_counts(dropna=False).rename("n").to_frame().assign(porcentaje=lambda x: x.n / len(df_parquet) * 100).round(2))

# Cohorte exploratoria

Se conserva `df_parquet` completo. Según el diccionario local, los códigos de
`discharge_disposition_id` 11, 19, 20 y 21 significan fallecimiento. Se
separan en `df_deceased` y no se eliminan del raw; `df_cohort` es la vista no
decedida usada en las gráficas y hallazgos porque un resultado de reingreso no
es interpretable del mismo modo en un episodio que termina en fallecimiento.
No se excluyen otras categorías, faltantes ni diagnósticos.

In [ ]:
death_codes = CONFIG["seleccion"]["disposiciones_fallecimiento"]   # [11, 19, 20, 21]
death_mask = df_parquet["discharge_disposition_id"].isin(death_codes)
df_deceased = df_parquet.loc[death_mask].copy()
df_cohort = df_parquet.loc[~death_mask].copy()

cohort_summary = pd.DataFrame([
    {"vista": "df_parquet (raw completo)", "filas": len(df_parquet), "% del raw": 100.0},
    {"vista": "df_deceased (separado)", "filas": len(df_deceased), "% del raw": len(df_deceased) / len(df_parquet) * 100},
    {"vista": "df_cohort (exploratoria)", "filas": len(df_cohort), "% del raw": len(df_cohort) / len(df_parquet) * 100},
])
display(cohort_summary.round(2))

# 2.3 · Explorar los datos

Para evitar un catálogo de gráficos ilegible se eligen cuatro magnitudes
representativas. Las figuras se calculan sobre `df_cohort`; las matrices y
cajas son asociaciones descriptivas y no implican causalidad.

In [ ]:
plot_cols = ["time_in_hospital", "num_lab_procedures", "num_medications", "number_inpatient"]

# Histogramas: distribución de cuatro magnitudes.
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.flat, plot_cols):
    sns.histplot(df_cohort[col], bins=20, color="#4472C4", ax=ax)
    ax.set_title(f"Histograma: {col}")
    ax.set_xlabel(col)
    ax.set_ylabel("episodios")
fig.tight_layout()
display(fig)
plt.close(fig)

# Barra de la variable objetivo.
target_order = ["NO", ">30", "<30"]
counts = df_cohort[target_col].value_counts().reindex(target_order, fill_value=0)
fig, ax = plt.subplots(figsize=(7, 4))
sns.barplot(x=counts.index, y=counts.values, hue=counts.index, legend=False, palette=["#70AD47", "#FFC000", "#C00000"], ax=ax)
ax.set_title("Distribución de readmitted en df_cohort")
ax.set_xlabel("resultado")
ax.set_ylabel("episodios")
fig.tight_layout()
display(fig)
plt.close(fig)

In [ ]:
# SPLOM: muestra fija para mantener la figura legible.
splom_data = df_cohort[plot_cols].sample(n=min(1000, len(df_cohort)), random_state=42)
axes = pd.plotting.scatter_matrix(splom_data, figsize=(12, 12), diagonal="hist", alpha=0.25, color="#2F5597")
fig = plt.gcf()
fig.suptitle("SPLOM de magnitudes numéricas (muestra aleatoria, semilla 42)", y=1.02)
fig.tight_layout()
display(fig)
plt.close(fig)

covariance = df_cohort[plot_cols].cov()
correlation = df_cohort[plot_cols].corr()
print("Matriz de covarianza:")
display(covariance.round(2))
print("Matriz de correlación de Pearson:")
display(correlation.round(2))
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(correlation, annot=True, fmt=".2f", cmap="vlag", center=0, ax=ax)
ax.set_title("Matriz de correlación de Pearson")
fig.tight_layout()
display(fig)
plt.close(fig)

In [ ]:
# Cajas por resultado.
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
for ax, col in zip(axes.flat, plot_cols):
    sns.boxplot(data=df_cohort, x=target_col, y=col, order=target_order, showfliers=False, color="#9DC3E6", ax=ax)
    ax.set_title(col)
    ax.set_xlabel("readmitted")
fig.tight_layout()
display(fig)
plt.close(fig)

# Barras agrupadas y apiladas: resultado según existencia de ingreso previo.
with_prior = df_cohort.assign(ingreso_previo=np.where(df_cohort["number_inpatient"] > 0, "Sí", "No"))
resultado_por_prior = pd.crosstab(with_prior["ingreso_previo"], with_prior[target_col], normalize="index").reindex(columns=target_order, fill_value=0)
resultado_por_prior = resultado_por_prior.reindex(["No", "Sí"])

ax = (resultado_por_prior * 100).plot(kind="bar", figsize=(8, 4), color=["#70AD47", "#FFC000", "#C00000"])
ax.set_title("Barras agrupadas: resultado por ingreso previo")
ax.set_xlabel("¿ingreso hospitalario previo?")
ax.set_ylabel("porcentaje dentro del grupo")
ax.legend(title="readmitted")
plt.xticks(rotation=0)
fig = ax.figure
fig.tight_layout()
display(fig)
plt.close(fig)

ax = (resultado_por_prior * 100).plot(kind="bar", stacked=True, figsize=(8, 4), color=["#70AD47", "#FFC000", "#C00000"])
ax.set_title("Barras apiladas: resultado por ingreso previo")
ax.set_xlabel("¿ingreso hospitalario previo?")
ax.set_ylabel("porcentaje dentro del grupo")
ax.legend(title="readmitted", bbox_to_anchor=(1.02, 1), loc="upper left")
plt.xticks(rotation=0)
fig = ax.figure
fig.tight_layout()
display(fig)
plt.close(fig)

### Tres hallazgos cuantificados

Cada hallazgo combina cifra, interpretación, consecuencia y evidencia. Son
asociaciones de esta muestra, no efectos clínicos ni promesas de reducción del
reingreso.

In [ ]:
porcentaje_temprano = df_cohort[target_col].eq("<30").mean() * 100
porcentaje_weight = df_cohort["weight"].isna().mean() * 100
prior_rates = with_prior.groupby("ingreso_previo", observed=False)[target_col].apply(lambda s: s.eq("<30").mean() * 100)
risk_without = prior_rates.get("No", np.nan)
risk_with = prior_rates.get("Sí", np.nan)
difference_pp = risk_with - risk_without

hallazgos_md = f"""
#### 🔹 Hallazgo 1 — Resultado minoritario

**`<30` representa el {porcentaje_temprano:.2f}%** de los {len(df_cohort):,} episodios de `df_cohort`.

> **Consecuencia:** la exactitud aislada puede ocultar esta distribución; es necesario definir una métrica de priorización antes de modelar.
>
> **Evidencia:** barra de `readmitted` y tabla del objetivo.

---

#### 🔹 Hallazgo 2 — Ausencia de `weight`

**Falta en el {porcentaje_weight:.2f}%** de la cohorte.

> **Consecuencia:** no conviene imputarlo automáticamente como una medición ordinaria; conviene evaluar un indicador de ausencia y su utilidad clínica.
>
> **Evidencia:** esquema de 50 variables y tabla de calidad.

---

#### 🔹 Hallazgo 3 — Ingresos previos

**`<30` aparece en el {risk_without:.2f}%** de episodios sin ingreso previo, frente a **{risk_with:.2f}%** con al menos uno — una diferencia descriptiva de **{difference_pp:.2f} puntos porcentuales**.

> **Consecuencia:** conservar esta señal para E3 y validar particiones por paciente. *No demuestra causalidad.*
>
> **Evidencia:** barras agrupadas/apiladas y cajas.
"""

display(Markdown(hallazgos_md))

# 2.4 · Verificar la calidad de los datos

Las tasas de calidad de esta sección se calculan sobre `df_parquet`, es decir,
sobre las **50 columnas y todos los episodios originales**. `df_cohort` se usa
solo para explorar. No se imputan valores, no se recortan atípicos y no se
borran filas en E2.

In [ ]:
# Faltantes y texto literal "None" en las 50 columnas originales.
missing_table = pd.DataFrame({
    "n faltantes": df_parquet.isna().sum(),
    "% faltantes": df_parquet.isna().mean() * 100,
    "cardinalidad no nula": df_parquet.nunique(dropna=True),
}).sort_values("% faltantes", ascending=False)
print(f"Celdas faltantes en las 50 columnas originales: {int(df_parquet.isna().sum().sum()):,} ({df_parquet.isna().mean().mean() * 100:.2f}% del total)")
print("La siguiente tabla y sus porcentajes usan siempre df_parquet (raw completo):")
display(missing_table.round(2))

literal_none = sum((df_parquet[c].astype("string") == "None").sum() for c in df_parquet.columns)
unknown_gender = int((df_parquet["gender"] == "Unknown/Invalid").sum())
print(f"Cadenas literales 'None': {literal_none}; gender='Unknown/Invalid': {unknown_gender} filas. Se documentan, no se convierten silenciosamente.")

In [ ]:
# Códigos administrativos relevantes según el diccionario concatenado.
quality_codes = []
for variable, grupo in mapping_table.groupby("variable"):
    observed = df_parquet[variable].value_counts(dropna=False)
    for _, row in grupo.iterrows():
        description = row["description"]
        lower = description.lower()
        code = row["codigo"]
        is_death = variable == "discharge_disposition_id" and code in death_codes
        is_special = is_death or any(word in lower for word in ["null", "not mapped", "not available", "unknown", "invalid"])
        if is_special:
            n = int(observed.get(code, 0))
            quality_codes.append({
                "variable": variable,
                "codigo": code,
                "description": description,
                "n en raw": n,
                "% en raw": n / len(df_parquet) * 100,
                "tratamiento": "separar fallecimientos para la vista; conservar raw" if is_death else "conservar y revisar",
            })
display(pd.DataFrame(quality_codes).round(2))

controls = pd.DataFrame([
    {"control": "filas exactas duplicadas", "n": int(df_parquet.duplicated().sum())},
    {"control": "encounter_id repetido", "n": int(df_parquet["encounter_id"].duplicated().sum())},
    {"control": "pacientes con más de un episodio", "n": int(df_parquet.loc[df_parquet["patient_nbr"].duplicated(keep=False), "patient_nbr"].nunique())},
    {"control": "filas de pacientes repetidos", "n": int(df_parquet["patient_nbr"].duplicated(keep=False).sum())},
])
print("Duplicados y repetición de identificadores en el raw:")
display(controls)

In [ ]:
# Atípicos descriptivos con la regla IQR; no se eliminan.
outlier_rows = []
for col in numericas_cols:
    serie = df_parquet[col]
    q1, q3 = serie.quantile([0.25, 0.75])
    iqr = q3 - q1
    fuera = (serie < q1 - 1.5 * iqr) | (serie > q3 + 1.5 * iqr)
    outlier_rows.append({
        "variable": col,
        "q1": q1,
        "q3": q3,
        "n fuera de IQR": int(fuera.sum()),
        "% fuera de IQR": fuera.mean() * 100,
        "decisión": "bandera descriptiva; no borrar ni recortar",
    })
display(pd.DataFrame(outlier_rows).set_index("variable").round(2))

# Representatividad básica de la cohorte exploratoria.
representacion = []
for col in ["age", "gender", "race"]:
    tabla = df_cohort.groupby(col, dropna=False)[target_col].agg(
        n="size", porcentaje_muestra=lambda s: s.size / len(df_cohort) * 100,
        porcentaje_menos_30=lambda s: s.eq("<30").mean() * 100,
    ).reset_index()
    tabla.insert(0, "variable", col)
    representacion.append(tabla)
print("Distribución descriptiva por edad, sexo y raza (df_cohort):")
display(pd.concat(representacion, ignore_index=True).round(2))

### Plan de calidad, limitaciones y cierre

- `weight`, `max_glu_serum`, `A1Cresult`, `medical_specialty`, `payer_code` y
`race` tienen ausencia importante. Se propone conservar indicadores de
ausencia y revisar utilidad con un experto; no imputar automáticamente en E2.
- Los valores fuera de IQR pueden ser válidos. Se mantienen y se revisarán en
E3; los códigos `Unknown/Invalid` se reportan sin eliminar episodios.
- `patient_nbr` se repite. Toda partición o evaluación posterior debe agrupar
por paciente para evitar fuga entre episodios.
- La muestra es histórica y proviene de hospitales de Estados Unidos, pero no
contiene el hospital. No se puede asegurar representatividad de otros centros,
países o periodos; tampoco se puede medir una tasa diaria porque no hay fechas.
- `<30` es un resultado observado, no una prueba de que el reingreso sea
prevenible. Faltan capacidad real de seguimiento, variables temporales,
validación clínica y criterios éticos para una intervención.

**Cierre:** los datos permiten continuar **con mitigaciones**: conservar el raw,
separar fallecimientos, documentar faltantes y códigos, agrupar por paciente y
revisar representatividad. Este notebook no entrena ni evalúa ningún modelo.

In [ ]:
quality_plan = pd.DataFrame([
    {"variable": "weight / resultados de laboratorio", "problema": "ausencia alta y categorías agrupadas", "siguiente paso": "indicador de ausencia y revisión clínica; no imputar automáticamente"},
    {"variable": "medical_specialty / payer_code / race", "problema": "cobertura diferencial", "siguiente paso": "auditar representación y uso ético"},
    {"variable": "diag_1, diag_2, diag_3", "problema": "faltantes y alta cardinalidad", "siguiente paso": "agrupar solo con aprobación clínica"},
    {"variable": "patient_nbr", "problema": "episodios repetidos por paciente", "siguiente paso": "particiones agrupadas; no deduplicar raw"},
    {"variable": "magnitudes numéricas", "problema": "ceros y valores fuera de IQR", "siguiente paso": "conservar y revisar antes de transformar"},
])
display(quality_plan)

# 2.5 · Publicar E2: contrato para E3 en MLflow, MinIO y GitHub

El análisis anterior se convierte en un **contrato**: un conjunto de archivos que E3 lee y verifica. Las cifras salen de las mismas tablas de este cuaderno (`schema`, `missing_table`, `quality_codes`, `controls`, `quality_plan`, hallazgos), así que el notebook y el contrato no pueden divergir.

| Destino | Qué queda | Para qué |
|---|---|---|
| **MLflow** | run `tipo=eda` con métricas de calidad, hallazgos, huella del crudo y figuras | raíz del linaje: E3, E4 y E5 lo citan en `run_eda` |
| **MinIO** | `datasets/crudo/<tag>/` (crudo exacto) · `datasets/eda/<tag>/` (contrato) · manifiesto · `CATALOGO.json` | E3 descarga **estos** bytes y verifica su md5 |
| **GitHub** | `reportes/e2/*` en la rama `e2/<tag>` con pull request (o commit directo) | el contrato versionado junto al código |

Después de aprobar el PR, en el repositorio se ejecuta **una sola orden**:

```
python tareas.py importar-e2 --tag crudo-v1
```

Esa orden baja el crudo desde MinIO, verifica su md5 contra `reportes/e2/descripcion.json`, lo registra en DVC con la etiqueta `crudo-v1` y lo publica. A partir de ahí, `dvc repro` construye E3.

In [ ]:
# --- Contrato E2 -> E3 (archivos deterministas: sin fechas) ----------------------
def guardar_json(nombre, datos):
    (SALIDA / nombre).write_text(json.dumps(datos, indent=2, ensure_ascii=False, sort_keys=True, default=float),
                                 encoding="utf-8")

r3 = lambda x: round(float(x), 3)

descripcion = {
    "fuente": "Diabetes 130-US Hospitals 1999-2008 (UCI, dataset 296)",
    "origen_parquet": ruta_parquet, "origen_mapping": ruta_ids_mapping,
    "archivo_crudo": "datos/crudo/diabetic_data.parquet", "md5_crudo": md5_crudo,
    "archivo_mapping": "datos/crudo/IDS_mapping.csv", "md5_mapping": md5_mapping,
    "crudo_version": TAG_CRUDO,
    "unidad_analisis": "episodio hospitalario (encounter_id)",
    "filas": len(df_parquet), "columnas": df_parquet.shape[1],
    "n_numericas": len(numericas_cols), "n_categoricas": len(categoricas_cols),
    "pacientes_unicos": int(df_parquet["patient_nbr"].nunique()),
    "objetivo": target_col,
    "distribucion_objetivo_pct": {k: r3(v * 100) for k, v in df_parquet[target_col].value_counts(normalize=True).items()},
    "prevalencia_lt30_pct": r3(df_parquet[target_col].eq("<30").mean() * 100),
    "filas_fallecimiento": len(df_deceased), "filas_cohorte": len(df_cohort),
    "tiene_fecha": False, "tiene_hospital": False,
}
calidad = {
    "celdas_faltantes": int(df_parquet.isna().sum().sum()),
    "pct_celdas_faltantes": r3(df_parquet.isna().mean().mean() * 100),
    "faltantes_pct": {c: r3(v) for c, v in missing_table["% faltantes"].items() if v > 0},
    "codigos_especiales": [{"variable": q["variable"], "codigo": int(q["codigo"]), "descripcion": q["description"],
                            "n": int(q["n en raw"]), "pct": r3(q["% en raw"])} for q in quality_codes],
    "genero_invalido": unknown_gender, "literal_none": int(literal_none),
    "constantes": [c for c in df_parquet.columns if df_parquet[c].nunique(dropna=True) <= 1],
    "duplicados": {fila["control"].replace(" ", "_"): int(fila["n"]) for _, fila in controls.iterrows()},
    "atipicos_iqr": {o["variable"]: {"n": o["n fuera de IQR"], "pct": r3(o["% fuera de IQR"])} for o in outlier_rows},
}
hallazgos = {"hallazgos": [
    {"id": "H1", "titulo": "Resultado minoritario", "valor": r3(porcentaje_temprano),
     "consecuencia": "métrica de priorización (Recall@K) declarada antes de modelar"},
    {"id": "H2", "titulo": "Ausencia de weight", "valor": r3(porcentaje_weight),
     "consecuencia": "no imputar; descartar o indicador de ausencia"},
    {"id": "H3", "titulo": "Ingresos previos (pp de diferencia)", "valor": r3(difference_pp),
     "sin_previo_pct": r3(risk_without), "con_previo_pct": r3(risk_with),
     "consecuencia": "señal para E3 y baseline B2; particiones por paciente"},
]}
etapa = ["E3 3.1/3.2", "E3 3.5 + auditoría E5", "E3 3.3", "E3 partición + CV E4", "E3 3.2"]
plan = {"plan": [dict(r, etapa_que_resuelve=e) for r, e in zip(quality_plan.to_dict("records"), etapa)]}

guardar_json("descripcion.json", descripcion)
guardar_json("calidad.json", calidad)
guardar_json("hallazgos.json", hallazgos)
guardar_json("plan_calidad.json", plan)
schema.round(3).to_csv(SALIDA / "esquema_e2.csv", index=False)
pd.concat(representacion, ignore_index=True).round(3).to_csv(SALIDA / "representatividad.csv", index=False)

# Figuras del contrato (las mismas lecturas de 2.3 y 2.4)
fig, ax = plt.subplots(figsize=(6, 3.8))
df_cohort[target_col].value_counts().reindex(["NO", ">30", "<30"]).plot(kind="bar", ax=ax, color=["#70AD47", "#FFC000", "#C00000"], rot=0)
ax.set_title("readmitted en la cohorte"); fig.tight_layout(); fig.savefig(SALIDA / "figuras/objetivo.png", dpi=110); plt.close(fig)
falt = missing_table["% faltantes"][missing_table["% faltantes"] > 0]
fig, ax = plt.subplots(figsize=(7, 4)); falt[::-1].plot(kind="barh", ax=ax, color="#C00000")
ax.axvline(50, ls="--", c="grey"); ax.set_title("% faltante por variable (raw)")
fig.tight_layout(); fig.savefig(SALIDA / "figuras/faltantes.png", dpi=110); plt.close(fig)
fig, ax = plt.subplots(figsize=(5, 3.8)); ax.bar(["sin ingreso previo", "con ingreso previo"], [risk_without, risk_with], color=["#4472C4", "#C00000"])
ax.set_ylabel("% <30"); ax.set_title("Hallazgo H3"); fig.tight_layout(); fig.savefig(SALIDA / "figuras/ingreso_previo.png", dpi=110); plt.close(fig)

display(pd.DataFrame({"archivo": sorted(str(p.relative_to(SALIDA)) for p in SALIDA.rglob("*") if p.is_file())}))

### MLflow: run de E2 (raíz del linaje)

In [ ]:
run_e2 = {}
if PUBLICAR_MLFLOW:
    mlflow = asegurar("mlflow")
    # Si el servidor guarda artefactos directo en MinIO, el cliente necesita estas variables
    os.environ.setdefault("MLFLOW_S3_ENDPOINT_URL", CONFIG["minio"]["endpoint"])
    if secreto("MINIO_ACCESS_KEY"):
        os.environ.setdefault("AWS_ACCESS_KEY_ID", secreto("MINIO_ACCESS_KEY"))
        os.environ.setdefault("AWS_SECRET_ACCESS_KEY", secreto("MINIO_SECRET_KEY"))
    mlflow.set_tracking_uri(CONFIG["mlflow"]["tracking_uri"])
    exp = mlflow.set_experiment(CONFIG["mlflow"]["experimento"])

    previos = mlflow.search_runs([exp.experiment_id], max_results=1,
                                 filter_string=f"tags.tipo = 'eda' and tags.md5_crudo = '{md5_crudo}' "
                                               f"and attributes.status = 'FINISHED'")
    if len(previos) and not FORZAR_NUEVO_RUN:
        run_id = previos.iloc[0]["run_id"]
        print(f"Este crudo ya tiene run E2 ({run_id[:8]}): se reutiliza (idempotente)")
    else:
        with mlflow.start_run(run_name=f"eda-{TAG_CRUDO}-{md5_crudo[:8]}") as r:
            run_id = r.info.run_id
            mlflow.set_tags({"tipo": "eda", "etapa": "E2", "proyecto": "P09", "md5_crudo": md5_crudo,
                             "crudo_version": TAG_CRUDO, "run_eda": run_id, "github_url": GITHUB_URL})
            mlflow.log_params({"fuente": descripcion["fuente"], "filas": descripcion["filas"],
                               "columnas": descripcion["columnas"], "unidad_analisis": descripcion["unidad_analisis"]})
            mlflow.log_metrics({"prevalencia_lt30_pct": descripcion["prevalencia_lt30_pct"],
                                "pct_celdas_faltantes": calidad["pct_celdas_faltantes"],
                                "filas_fallecimiento": descripcion["filas_fallecimiento"],
                                **{f"faltante_pct_{c}": v for c, v in calidad["faltantes_pct"].items()},
                                **{f"hallazgo_{h['id']}": h["valor"] for h in hallazgos["hallazgos"]}})
            ds = mlflow.data.from_pandas(df_parquet.head(2000), name=f"diabetic_data-{TAG_CRUDO}", targets=target_col,
                                         source=f"s3://{CONFIG['minio']['bucket']}/{CONFIG['minio']['prefijo_datasets']}/crudo/{TAG_CRUDO}/diabetic_data.parquet")
            mlflow.log_input(ds, context="eda")
            mlflow.log_artifacts(str(SALIDA), artifact_path="e2")
        print(f"Run E2 creado: {run_id[:8]}")
    run_e2 = {"run_id": run_id, "experiment_id": exp.experiment_id, "experimento": exp.name,
              "tracking_uri": CONFIG["mlflow"]["tracking_uri"], "md5_crudo": md5_crudo, "crudo_version": TAG_CRUDO,
              "url": f"{CONFIG['mlflow']['tracking_uri']}/#/experiments/{exp.experiment_id}/runs/{run_id}"}
    guardar_json("mlflow_run.json", run_e2)
    print(run_e2["url"])
else:
    print("MLflow omitido (PUBLICAR_MLFLOW = False)")

### MinIO: crudo exacto y contrato, organizados por versión

In [ ]:
if PUBLICAR_MINIO:
    if not (secreto("MINIO_ACCESS_KEY") and secreto("MINIO_SECRET_KEY")):
        raise RuntimeError("PUBLICAR_MINIO = True pero faltan credenciales: " +
                           str({k: DIAG_SECRETOS.get(k) for k in ("MINIO_ACCESS_KEY", "MINIO_SECRET_KEY")}))
    boto3 = asegurar("boto3")
    from botocore.config import Config as BotoConfig
    s3 = boto3.client("s3", endpoint_url=CONFIG["minio"]["endpoint"], region_name="us-east-1",
                      aws_access_key_id=secreto("MINIO_ACCESS_KEY"), aws_secret_access_key=secreto("MINIO_SECRET_KEY"),
                      config=BotoConfig(signature_version="s3v4", s3={"addressing_style": "path"},
                                        connect_timeout=10, read_timeout=300, retries={"max_attempts": 3}))
    B, P = CONFIG["minio"]["bucket"], CONFIG["minio"]["prefijo_datasets"]
    if B not in {b["Name"] for b in s3.list_buckets().get("Buckets", [])}:
        s3.create_bucket(Bucket=B)

    archivos = [("crudo", "diabetic_data.parquet", bytes_parquet), ("crudo", "IDS_mapping.csv", bytes_mapping)]
    archivos += [("eda", p.relative_to(SALIDA).as_posix(), p.read_bytes()) for p in sorted(SALIDA.rglob("*")) if p.is_file()]
    manifiesto = {"tag": TAG_CRUDO, "capas": ["crudo", "eda"], "github_url": GITHUB_URL,
                  "procedencia": {"run_eda": run_e2.get("run_id"), "md5_crudo": md5_crudo}, "archivos": []}
    for capa, nombre, contenido in archivos:
        clave = f"{P}/{capa}/{TAG_CRUDO}/{nombre}"
        s3.put_object(Bucket=B, Key=clave, Body=contenido, Metadata={"md5": hashlib.md5(contenido).hexdigest()})
        manifiesto["archivos"].append({"capa": capa, "clave_s3": clave, "bytes": len(contenido),
                                       "md5": hashlib.md5(contenido).hexdigest()})
        print(f"  ↑ s3://{B}/{clave}")
    s3.put_object(Bucket=B, Key=f"{P}/manifiestos/{TAG_CRUDO}.json", ContentType="application/json",
                  Body=json.dumps(manifiesto, indent=2, ensure_ascii=False).encode())

    try:   # catálogo: índice de todas las versiones (mismo formato que usa E3)
        catalogo = json.loads(s3.get_object(Bucket=B, Key=f"{P}/CATALOGO.json")["Body"].read())
    except Exception:
        catalogo = {"proyecto": "P09", "github_url": GITHUB_URL, "versiones": []}
    catalogo["versiones"] = [v for v in catalogo["versiones"] if v["tag"] != TAG_CRUDO] + [{
        "tag": TAG_CRUDO, "capas": ["crudo", "eda"], "md5_tabla_principal": md5_crudo,
        "filas_tabla_principal": len(df_parquet), "run_eda": run_e2.get("run_id"),
        "manifiesto": f"{P}/manifiestos/{TAG_CRUDO}.json"}]
    catalogo["versiones"].sort(key=lambda v: v["tag"])
    s3.put_object(Bucket=B, Key=f"{P}/CATALOGO.json", ContentType="application/json",
                  Body=json.dumps(catalogo, indent=2, ensure_ascii=False).encode())
    display(pd.DataFrame(catalogo["versiones"]))
    print(f"✓ MinIO: {len(archivos)} archivos en s3://{B}/{P}/(crudo|eda)/{TAG_CRUDO}/")
else:
    print("MinIO desactivado (PUBLICAR_MINIO = False)")

### GitHub: el contrato entra al repositorio por pull request

In [ ]:
def gh(metodo, ruta, cuerpo=None, tolerar=()):
    """Llamada mínima a la API REST de GitHub (sin git ni clonar el repositorio).
    `tolerar`: códigos HTTP que se devuelven como None en vez de lanzar error."""
    api = os.environ.get("P09_GITHUB_API", "https://api.github.com")    # sobrescribible para pruebas
    url = f"{api}/repos/{slug_github(GITHUB_URL)}{ruta}"
    req = urllib.request.Request(url, method=metodo,
                                 data=json.dumps(cuerpo).encode() if cuerpo is not None else None,
                                 headers={"Authorization": f"Bearer {secreto('GITHUB_TOKEN')}",
                                          "Accept": "application/vnd.github+json", "X-GitHub-Api-Version": "2022-11-28"})
    try:
        with urllib.request.urlopen(req, timeout=60) as r:
            return json.loads(r.read() or b"{}")
    except urllib.error.HTTPError as e:
        if e.code in tolerar:
            return None
        detalle = ""
        try:
            detalle = (json.loads(e.read() or b"{}") or {}).get("message", "")
        except Exception:
            pass
        e.contexto = f"{metodo} {url} -> HTTP {e.code} {detalle}"
        raise


pr_url = None
if PUBLICAR_GITHUB:
    if not (secreto("GITHUB_TOKEN") and slug_github(GITHUB_URL) and "CAMBIAR-USUARIO" not in GITHUB_URL):
        raise RuntimeError(f"PUBLICAR_GITHUB = True pero GITHUB_TOKEN: {DIAG_SECRETOS.get('GITHUB_TOKEN')}")
    sub = f"{GITHUB_SUBCARPETA.strip('/')}/" if GITHUB_SUBCARPETA.strip("/") else ""
    archivos = {f"{sub}reportes/e2/{p.relative_to(SALIDA).as_posix()}": p.read_bytes()
                for p in sorted(SALIDA.rglob("*")) if p.is_file()}

    # La rama de trabajo: la de PARÁMETROS si existe; si no, la rama por defecto del repo
    repo = gh("GET", "")
    rama_base = GITHUB_RAMA
    ref = gh("GET", f"/git/ref/heads/{rama_base}", tolerar=(404, 409))
    if ref is None and repo.get("default_branch") and repo["default_branch"] != GITHUB_RAMA:
        rama_base = repo["default_branch"]
        ref = gh("GET", f"/git/ref/heads/{rama_base}", tolerar=(404, 409))
        if ref:
            print(f"⚠  La rama '{GITHUB_RAMA}' no existe; se usa la rama por defecto del repo: '{rama_base}'. "
                  f"Ajuste GITHUB_RAMA en PARÁMETROS.")
    repo_vacio = ref is None

    if repo_vacio:   # repositorio sin ningún commit: se crea el primero
        print(f"El repositorio no tiene commits en '{rama_base}': se crea el commit inicial.")
        arbol = gh("POST", "/git/trees", {"tree": [
            {"path": ruta, "mode": "100644", "type": "blob",
             "sha": gh("POST", "/git/blobs", {"content": base64.b64encode(b).decode(), "encoding": "base64"})["sha"]}
            for ruta, b in archivos.items()]})["sha"]
        base = None
    else:
        base = ref["object"]["sha"]
        arbol_base = gh("GET", f"/git/commits/{base}")["tree"]["sha"]
        arbol = gh("POST", "/git/trees", {"base_tree": arbol_base, "tree": [
            {"path": ruta, "mode": "100644", "type": "blob",
             "sha": gh("POST", "/git/blobs", {"content": base64.b64encode(b).decode(), "encoding": "base64"})["sha"]}
            for ruta, b in archivos.items()]})["sha"]
    mensaje = (f"e2: contrato de comprensión de datos {TAG_CRUDO}\n\n"
               f"{len(df_parquet):,} filas · <30 = {descripcion['prevalencia_lt30_pct']} % · md5 {md5_crudo}\n"
               f"MLflow run: {run_e2.get('url', 'no publicado')}\n"
               f"MinIO: s3://{CONFIG['minio']['bucket']}/{CONFIG['minio']['prefijo_datasets']}/(crudo|eda)/{TAG_CRUDO}/")
    commit = gh("POST", "/git/commits",
                {"message": mensaje, "tree": arbol, "parents": [base] if base else []})["sha"]

    if repo_vacio or GITHUB_MODO == "directo":
        # Sin commits previos no se puede abrir un PR: el primero va a la rama base
        if repo_vacio:
            gh("POST", "/git/refs", {"ref": f"refs/heads/{rama_base}", "sha": commit})
        else:
            gh("PATCH", f"/git/refs/heads/{rama_base}", {"sha": commit})
        print(f"✓ commit {commit[:8]} en {rama_base}" + (" (commit inicial del repositorio)" if repo_vacio else ""))
    else:
        rama = f"e2/{TAG_CRUDO}"
        if gh("POST", "/git/refs", {"ref": f"refs/heads/{rama}", "sha": commit}, tolerar=(422,)) is None:
            gh("PATCH", f"/git/refs/heads/{rama}", {"sha": commit, "force": True})   # re-ejecución
        pr = gh("POST", "/pulls", {"title": f"E2 · contrato {TAG_CRUDO}", "head": rama, "base": rama_base,
                                   "body": mensaje + "\n\nAl integrar: `python tareas.py importar-e2 --tag " + TAG_CRUDO + "`"},
                tolerar=(422,))
        if pr is None:            # ya existía un PR abierto para esa rama
            abiertos = gh("GET", f"/pulls?head={slug_github(GITHUB_URL).split('/')[0]}:{rama}&state=open")
            pr = abiertos[0] if abiertos else {"html_url": f"{repo['html_url']}/compare/{rama_base}...{rama}"}
        pr_url = pr["html_url"]
        print(f"✓ rama {rama} (sobre {rama_base}) · pull request: {pr_url}")
    del archivos
else:
    print("GitHub desactivado (PUBLICAR_GITHUB = False).")
    print(f"Alternativa manual: subir el contenido de {SALIDA} a reportes/e2/ del repositorio.")

### Resumen de la publicación

In [ ]:
en_celda = [n for n in ("MINIO_ACCESS_KEY", "MINIO_SECRET_KEY", "GITHUB_TOKEN")
            if DIAG_SECRETOS.get(n, "").startswith("celda CREDENCIALES")]
if en_celda:
    print("⚠  Este cuaderno tiene secretos escritos en la celda CREDENCIALES: " + ", ".join(en_celda))
    print("   Antes de subirlo a GitHub o compartirlo: vaciar esos valores (dejar "").")
    print("   Si ya se subió o se compartió, REVOCAR el token y ROTAR la clave de MinIO: borrarlo no basta.")

resumen = pd.DataFrame([
    ["crudo", f"{len(df_parquet):,} filas × {df_parquet.shape[1]} col · md5 {md5_crudo}"],
    ["versión", TAG_CRUDO],
    ["MLflow", run_e2.get("url", "no publicado")],
    ["MinIO", f"s3://{CONFIG['minio']['bucket']}/{CONFIG['minio']['prefijo_datasets']}/(crudo|eda)/{TAG_CRUDO}/"
              if PUBLICAR_MINIO else "no publicado"],
    ["GitHub", pr_url or ("commit directo" if GITHUB_MODO == "directo" and PUBLICAR_GITHUB else "no publicado")],
    ["siguiente paso (repo)", f"aprobar el PR y ejecutar: python tareas.py importar-e2 --tag {TAG_CRUDO}"],
], columns=["elemento", "valor"])
display(resumen)

**Cierre de E2:**
- Los datos permiten continuar **con mitigaciones**: no tienen fecha ni hospital, y faltan variables de adherencia.
- El crudo queda identificado por su md5, descrito en el contrato y registrado en MLflow, MinIO y GitHub.
- E3 no vuelve a leer Drive: parte de los bytes publicados aquí y se detiene si no coinciden con el contrato.